In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets evaluate accelerate rouge_score sentencepiece pandas numpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [3]:
import os
import json
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup
)
import evaluate
from tqdm.auto import tqdm

In [21]:
EXPERIMENT_NAME = "distill_from_base_raw"
ABLATION_TYPE = "raw"   # Options: raw | coref | textrank | coref+tr

DATA_PATH = f"/content/drive/MyDrive/data/ablation/{ABLATION_TYPE}.csv"

TEACHER_MODEL_NAME = "facebook/bart-large-cnn"
STUDENT_MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

OUTPUT_DIR = f"/content/drive/MyDrive/data/distill/{EXPERIMENT_NAME}_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cpu


In [6]:
from sklearn.model_selection import train_test_split

full_df = pd.read_csv(DATA_PATH)
print("Full dataset shape:", full_df.shape)

# Fixed split — same random_state across all ablation files for fair comparison
train_df, temp_df = train_test_split(full_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Full dataset shape: (446, 3)
Train shape: (312, 3)
Validation shape: (67, 3)
Test shape: (67, 3)


,doc,original_text,reference_summary
0,Open-Source Terms,npm services may provide information and softw...,the service does not guarantee accuracy or rel...
1,Terms of Service and License Agreement,by uploading or entering any user content you ...,this service can use your content for all thei...
2,Terms of Use,if you seek to file a legal claim against us y...,the court of law governing the terms is in san...
3,Privacy Policy,if this policy is substantively updated we wil...,conditions may change but your continued accep...
4,Privacy Policy,open book publishers will not send you email t...,you can opt out of promotional communications.


In [7]:
def find_first_existing(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

INPUT_CANDIDATES = ["input_text", "clean_text", "original_text", "text", "source_text"]
TARGET_CANDIDATES = ["target_summary", "clean_summary", "reference_summary", "case_text", "summary"]

input_col = find_first_existing(train_df, INPUT_CANDIDATES)
target_col = find_first_existing(train_df, TARGET_CANDIDATES)

print("Detected input column:", input_col)
print("Detected target column:", target_col)

if input_col is None or target_col is None:
    raise ValueError("Could not detect input or target summary columns.")

Detected input column: original_text
Detected target column: reference_summary


In [8]:
train_df = train_df[[input_col, target_col]].dropna().reset_index(drop=True)
val_df = val_df[[input_col, target_col]].dropna().reset_index(drop=True)
test_df = test_df[[input_col, target_col]].dropna().reset_index(drop=True)

train_df = train_df.rename(columns={input_col: "input_text", target_col: "target_summary"})
val_df = val_df.rename(columns={input_col: "input_text", target_col: "target_summary"})
test_df = test_df.rename(columns={input_col: "input_text", target_col: "target_summary"})

display(train_df.head())

,input_text,target_summary
0,npm services may provide information and softw...,the service does not guarantee accuracy or rel...
1,by uploading or entering any user content you ...,this service can use your content for all thei...
2,if you seek to file a legal claim against us y...,the court of law governing the terms is in san...
3,if this policy is substantively updated we wil...,conditions may change but your continued accep...
4,open book publishers will not send you email t...,you can opt out of promotional communications.


In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 312
    })
    validation: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 67
    })
    test: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 67
    })
})

In [10]:
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 64

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

tokenized_datasets

Map:   0%|          | 0/312 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 312
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 67
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 67
    })
})

In [12]:
teacher_tmp = AutoModelForSeq2SeqLM.from_pretrained(TEACHER_MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=teacher_tmp)
del teacher_tmp

train_loader = DataLoader(
    tokenized_datasets["train"],
    batch_size=2,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    tokenized_datasets["test"],
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [13]:
teacher_model = AutoModelForSeq2SeqLM.from_pretrained(TEACHER_MODEL_NAME).to(device)
student_model = AutoModelForSeq2SeqLM.from_pretrained(STUDENT_MODEL_NAME).to(device)

teacher_model.eval()
print("Teacher loaded from:", TEACHER_MODEL_NAME)
print("Student loaded from:", STUDENT_MODEL_NAME)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Teacher loaded from: facebook/bart-large-cnn
Student loaded from: sshleifer/distilbart-cnn-12-6


In [14]:
class DistillationLoss(nn.Module):
    def __init__(self, temperature=2.0, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=-100)
        self.kl_loss = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, labels):
        """
        student_logits: [batch, seq_len, vocab]
        teacher_logits: [batch, seq_len, vocab]
        labels: [batch, seq_len]
        """
        # 1. Standard supervised loss with gold labels
        ce = self.ce_loss(
            student_logits.view(-1, student_logits.size(-1)),
            labels.view(-1)
        )

        # 2. Distillation loss with teacher logits
        T = self.temperature

        student_log_probs = F.log_softmax(student_logits / T, dim=-1)
        teacher_probs = F.softmax(teacher_logits / T, dim=-1)

        kl = self.kl_loss(student_log_probs, teacher_probs) * (T ** 2)

        # 3. Weighted combination
        loss = self.alpha * ce + (1 - self.alpha) * kl
        return loss, ce.detach(), kl.detach()

In [15]:
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
TEMPERATURE = 2.0
ALPHA = 0.5

optimizer = torch.optim.AdamW(student_model.parameters(), lr=LEARNING_RATE)

num_training_steps = NUM_EPOCHS * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

distill_criterion = DistillationLoss(temperature=TEMPERATURE, alpha=ALPHA)

In [16]:
def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

best_val_loss = float("inf")
history = []

for epoch in range(NUM_EPOCHS):
    student_model.train()
    total_loss = 0
    total_ce = 0
    total_kl = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for batch in progress_bar:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_outputs = teacher_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

        student_outputs = student_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss, ce, kl = distill_criterion(
            student_outputs.logits,
            teacher_outputs.logits,
            batch["labels"]
        )

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_ce += ce.item()
        total_kl += kl.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "ce": f"{ce.item():.4f}",
            "kl": f"{kl.item():.4f}"
        })

    avg_train_loss = total_loss / len(train_loader)
    avg_train_ce = total_ce / len(train_loader)
    avg_train_kl = total_kl / len(train_loader)

    # Validation loss
    student_model.eval()
    val_loss_total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = move_batch_to_device(batch, device)

            teacher_outputs = teacher_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            student_outputs = student_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )

            val_loss, _, _ = distill_criterion(
                student_outputs.logits,
                teacher_outputs.logits,
                batch["labels"]
            )

            val_loss_total += val_loss.item()

    avg_val_loss = val_loss_total / len(val_loader)

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_ce": avg_train_ce,
        "train_kl": avg_train_kl,
        "val_loss": avg_val_loss
    })

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f} | CE: {avg_train_ce:.4f} | KL: {avg_train_kl:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")

    # Save best checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        student_model.save_pretrained(f"{OUTPUT_DIR}/best_student_model")
        tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_student_model")
        print("Saved best student checkpoint.")

Epoch 1/2:   0%|          | 0/156 [00:00<?, ?it/s]


Epoch 1
Train Loss: 11.1335 | CE: 3.6321 | KL: 18.6350
Val Loss: 8.0069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best student checkpoint.


Epoch 2/2:   0%|          | 0/156 [00:00<?, ?it/s]


Epoch 2
Train Loss: 7.3736 | CE: 3.0244 | KL: 11.7227
Val Loss: 7.2209


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best student checkpoint.


In [17]:
history_df = pd.DataFrame(history)
history_df.to_csv(f"{OUTPUT_DIR}/distillation_history.csv", index=False)
history_df

,epoch,train_loss,train_ce,train_kl,val_loss
0,1,11.133534,3.632063,18.635004,8.006949
1,2,7.373575,3.024420,11.722730,7.220923


In [18]:
student_model = AutoModelForSeq2SeqLM.from_pretrained(f"{OUTPUT_DIR}/best_student_model").to(device)
student_model.eval()

Loading weights:   0%|          | 0/356 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [19]:
rouge = evaluate.load("rouge")

def generate_summaries(model, dataloader, tokenizer, max_length=64, num_beams=4, length_penalty=1.0):
    all_preds = []
    all_refs = []

    for batch in tqdm(dataloader, desc="Generating summaries"):
        batch = move_batch_to_device(batch, device)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=max_length,
                num_beams=num_beams,
                length_penalty=length_penalty,
                early_stopping=True
            )

        preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        labels = batch["labels"].clone()
        labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)
        refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

        all_preds.extend([p.strip() for p in preds])
        all_refs.extend([r.strip() for r in refs])

    return all_preds, all_refs

val_preds, val_refs = generate_summaries(student_model, val_loader, tokenizer)
val_rouge = rouge.compute(predictions=val_preds, references=val_refs)

test_preds, test_refs = generate_summaries(student_model, test_loader, tokenizer)
test_rouge = rouge.compute(predictions=test_preds, references=test_refs)

print("Validation ROUGE:", val_rouge)
print("Test ROUGE:", test_rouge)

Generating summaries:   0%|          | 0/34 [00:00<?, ?it/s]

Generating summaries:   0%|          | 0/34 [00:00<?, ?it/s]

Validation ROUGE: {'rouge1': np.float64(0.20834224624624736), 'rouge2': np.float64(0.06819309160810381), 'rougeL': np.float64(0.1546293809638843), 'rougeLsum': np.float64(0.15563191479627786)}
Test ROUGE: {'rouge1': np.float64(0.19832329032312698), 'rouge2': np.float64(0.06842867596506519), 'rougeL': np.float64(0.1545356010333101), 'rougeLsum': np.float64(0.15523366894031781)}


In [22]:
# Extended evaluation: same distilled model, evaluated on all 4 ablation test inputs.
# Uses random_state=42 (same as Cell 4) so the SAME documents are in test across ablations.

ABLATION_TYPES = ["raw", "coref", "textrank", "coref+tr"]
ablation_results = {}

for ab_type in ABLATION_TYPES:
    print(f"\n========== Evaluating on '{ab_type}' input ==========")

    # 1. Load and split with same random_state=42 → same test indices across ablations
    ab_full_df = pd.read_csv(f"/content/drive/MyDrive/data/ablation/{ab_type}.csv")
    _, ab_temp_df = train_test_split(ab_full_df, test_size=0.30, random_state=42)
    _, ab_test_df = train_test_split(ab_temp_df, test_size=0.50, random_state=42)
    ab_test_df = ab_test_df.reset_index(drop=True)

    # 2. Standardize columns via the same auto-detection as Cell 5
    in_col = find_first_existing(ab_test_df, INPUT_CANDIDATES)
    tg_col = find_first_existing(ab_test_df, TARGET_CANDIDATES)
    ab_test_df = ab_test_df[[in_col, tg_col]].dropna().rename(
        columns={in_col: "input_text", tg_col: "target_summary"}
    )

    # 3. Tokenize and build DataLoader
    ab_dataset = Dataset.from_pandas(ab_test_df, preserve_index=False)
    ab_tokenized = ab_dataset.map(
        preprocess_function, batched=True, remove_columns=ab_dataset.column_names
    )
    ab_loader = DataLoader(
        ab_tokenized, batch_size=2, shuffle=False, collate_fn=data_collator
    )

    # 4. Generate + ROUGE (reusing generate_summaries from Cell 17)
    preds, refs = generate_summaries(student_model, ab_loader, tokenizer)
    ab_rouge = rouge.compute(predictions=preds, references=refs)

    ablation_results[ab_type] = {
        "rouge1":    float(ab_rouge["rouge1"]),
        "rouge2":    float(ab_rouge["rouge2"]),
        "rougeL":    float(ab_rouge["rougeL"]),
        "rougeLsum": float(ab_rouge["rougeLsum"]),
    }

    # Save per-ablation sample outputs for qualitative analysis
    sample_df = pd.DataFrame({
        "generated_summary": preds[:5],
        "reference_summary": refs[:5]
    })
    safe_name = ab_type.replace("+", "_plus_")
    sample_df.to_csv(f"{OUTPUT_DIR}/sample_outputs_{safe_name}.csv", index=False)

    print(f"[{ab_type}] ROUGE: {ab_rouge}")

# Compile into a single comparison table
ablation_df = pd.DataFrame(ablation_results).T
ablation_df.to_csv(f"{OUTPUT_DIR}/distilled_ablation_rouge.csv")

print("\n========== Distilled row of the ablation table ==========")
display(ablation_df)


========== Evaluating on 'raw' input ==========


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Generating summaries:   0%|          | 0/34 [00:00<?, ?it/s]

[raw] ROUGE: {'rouge1': np.float64(0.19832329032312698), 'rouge2': np.float64(0.06842867596506519), 'rougeL': np.float64(0.1545356010333101), 'rougeLsum': np.float64(0.15523366894031781)}

========== Evaluating on 'coref' input ==========


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

...

[textrank] ROUGE: {'rouge1': np.float64(0.1974845721414214), 'rouge2': np.float64(0.06355456487660743), 'rougeL': np.float64(0.152159143444664), 'rougeLsum': np.float64(0.1525760342665283)}

========== Evaluating on 'coref+tr' input ==========


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Generating summaries:   0%|          | 0/34 [00:00<?, ?it/s]

[coref+tr] ROUGE: {'rouge1': np.float64(0.19392199210973893), 'rouge2': np.float64(0.061859665160784325), 'rougeL': np.float64(0.15284220007605848), 'rougeLsum': np.float64(0.15317441669877657)}

========== Distilled row of the ablation table ==========


,rouge1,rouge2,rougeL,rougeLsum
raw,0.198323,0.068429,0.154536,0.155234
coref,0.193815,0.064276,0.151671,0.152227
textrank,0.197485,0.063555,0.152159,0.152576
coref+tr,0.193922,0.061860,0.152842,0.153174


In [23]:
metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "teacher_model": TEACHER_MODEL_NAME,
    "student_model": STUDENT_MODEL_NAME,
    "temperature": TEMPERATURE,
    "alpha": ALPHA,
    "num_epochs": NUM_EPOCHS,
    "trained_on_ablation": ABLATION_TYPE,
    "validation_rouge": val_rouge,
    "test_rouge": test_rouge,
    "ablation_rouge": ablation_results
}

with open(f"{OUTPUT_DIR}/distillation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)

print("Saved distillation metrics.")

Saved distillation metrics.


In [24]:
sample_output_df = pd.DataFrame({
    "generated_summary": test_preds[:10],
    "reference_summary": test_refs[:10]
})

sample_output_df.to_csv(f"{OUTPUT_DIR}/sample_outputs.csv", index=False)
sample_output_df

,generated_summary,reference_summary
0,the laws of california u s a excluding califCa...,the court of law governing the terms is in cal...
1,since the like button is a little piece of fac...,this service tracks you on other websites.
2,you agree not to upload post or otherwise tran...,users agree not to submit libelous harassing o...
3,you re responsible for your conduct your stuff...,you have to respect the content of others via ...
4,technical data which may include URL informati...,the service may use tracking pixels web beacon...
5,tldr permits access to the sites subject to th...,by using tldrlegal you agree to these terms. w...
6,any other information requested such as your r...,you agree to provide your full legal name when...
7,you agree that i the service shall be deemed s...,the jurisdiction is california.
8,if you choose to create an account then the mi...,the service allows you to use pseudonyms.
9,22 4 if any provision of these terms is held t...,invalidity of any portion of the terms of serv...
